In [ ]:
%load_ext autoreload
%autoreload 2
import os
import numpy as np
import torch
from src.constants import BASEDIR
import hydra
from hydra import initialize_config_dir, compose
from src.data.collators import CustomDataCollator
from src.data.datasets import load_protein_dataset

os.chdir(BASEDIR)

DATASET_NAME = "foldseek_representatives_interleaved"
MAX_TOKENS = 8192
MASK_BELOW_PLDDT = 80

TODO: figure out how to change namespace config gets loaded to (@ - package)

critical for example data.

In [ ]:
with initialize_config_dir(os.path.join(BASEDIR, "configs")):
    cfg = compose(config_name=f"data/dataset/{DATASET_NAME}")
    tokenizer_cfg = compose(
        config_name="tokenizer/profam",
    )

In [ ]:
dataset_cfg = hydra.utils.instantiate(cfg.data.dataset, _convert_="partial")
tokenizer = hydra.utils.instantiate(tokenizer_cfg.tokenizer)

In [ ]:
dataset_cfg

In [ ]:
feature_names = ["input_ids", "attention_mask", "labels", "seq_pos", "ds_name", "identifier", "total_num_sequences", "aa_mask", "coords", "coords_mask", "interleaved_coords_mask", "structure_mask", "plddts"]
data = load_protein_dataset(
    dataset_cfg,
    tokenizer,
    dataset_name=DATASET_NAME,
    data_dir="data/example_data",
    max_tokens_per_example=2048,
    feature_names=feature_names,
    shuffle=False,
    return_format="numpy"
)

In [ ]:
collator = CustomDataCollator(tokenizer, feature_names=None)

In [ ]:
%%timeit -n 3 -r 1
iterator = iter(data)
example = next(iterator)
batch = collator([example]*10)

In [ ]:
iterator = iter(data)
example = next(iterator)

In [ ]:
example["coords"]

In [ ]:
example["aa_mask"]

In [ ]:
example["structure_mask"]

In [ ]:
np.array(example["interleaved_coords_mask"]).any((-1,-2)).sum()

In [ ]:
example["aa_mask"].sum()

In [ ]:
example["plddts"][2:50]

In [ ]:
example["structure_mask"].sum()

In [ ]:
example["high_plddt_mask"].sum()

In [ ]:
example.keys()

In [ ]:
batch = {k: v[None] for k, v in example.items() if torch.is_tensor(v)}

In [ ]:
example["input_ids"][:400]

In [ ]:
example["coords"][:400]

In [ ]:
example["plddt_mask"]

In [ ]:
example["structure_mask"]

In [ ]:
example["input_ids"]

TODO: add pymol visualisation.